# Classification

Recipe Classification on the time it takes to make them

In [8]:
import pandas as pd
import ast
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
import difflib
from rapidfuzz import process, fuzz

ps = PorterStemmer()

In [9]:
data  = pd.read_csv('../../data/RAW_recipes.csv')
data.set_index('id', inplace=True)
columns = ["tags", "steps", "ingredients", "nutrition"]

for i in columns:
    data[i] = data[i].apply(ast.literal_eval)

data.drop(columns=["contributor_id", "submitted"], inplace=True, errors="ignore")
data.dropna(subset=["name"], inplace=True)
data.head()

,name,minutes,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
id,,,,,,,,,
137739,arriba baked winter squash mexican style,55,"[60-minutes-or-less, time-to-make, course, mai...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"[make a choice and proceed with recipe, depend...",autumn is my favorite time of year to cook! th...,"[winter squash, mexican seasoning, mixed spice...",7
31490,a bit different breakfast pizza,30,"[30-minutes-or-less, time-to-make, course, mai...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"[preheat oven to 425 degrees f, press dough in...",this recipe calls for the crust to be prebaked...,"[prepared pizza crust, sausage patty, eggs, mi...",6
112140,all in the kitchen chili,130,"[time-to-make, course, preparation, main-dish,...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"[brown ground beef in large pot, add chopped o...",this modified version of 'mom's' chili was a h...,"[ground beef, yellow onions, diced tomatoes, t...",13
59389,alouette potatoes,45,"[60-minutes-or-less, time-to-make, course, mai...","[368.1, 17.0, 10.0, 2.0, 14.0, 8.0, 20.0]",11,[place potatoes in a large pot of lightly salt...,"this is a super easy, great tasting, make ahea...","[spreadable cheese with garlic and herbs, new ...",11
44061,amish tomato ketchup for canning,190,"[weeknight, time-to-make, course, main-ingredi...","[352.9, 1.0, 337.0, 23.0, 3.0, 0.0, 28.0]",5,"[mix all ingredients& boil for 2 1 / 2 hours ,...",my dh's amish mother raised him on this recipe...,"[tomato juice, apple cider vinegar, sugar, sal...",8


## Small preprocessing
### Stop-word removing

# TODO 
=> Stop words on descriptions and steps

In [10]:
nltk.download('stopwords')
nltk.download('wordnet')

# Initialize the lemmatizer and stopwords list
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
additional = {
    "minutes", "easiest", "ever", "aww", "i", "can", "t", "believe", "it", "s", "stole", "the", "idea", "from","mirj", "andrea", " s ", "andreas",
    "viestad", "andes", "andersen", "an", "ana", "amy", "2 ww points", "on demand", "anelia", "amazing",
    "ashley", "ashton", "amazing", "make", "house", "smell", "malcolm", "amazingly", "killer", "perfect",
    "addictive", "leave", "u", "licking", "ur", "finger", "clean", "th", "recipe", "special", "time", "favorite",
    "aunt", "jane", "soft", "and", "moist", "licking", "famous", "non fruitcake", "true", "later",
    "nonbeliever", "believer", "comfort", "ultimate", "lover", "love", "easy", "ugly", "cc", "uncle", "bill", "tyler",
    "unbelievably", "unbelievable", "healthy", "fat", "free", "un", "melt", "mouth", "ummmmm", "umm", "ummmy", "nummy", "ummmm", "unattended",
    "unbaked", "ultra", "ultimately", "yummy", "rich", "quick", "rachael", "ray", "fail", "party", "florence",
    "fast", "light", "low", "carb", "snack", "wedding", "anniversary", "anne", "marie", "annemarie", "annette", "funicello", "syms",
    "byrn", "mike", "willan", "summer", "autumn", "winter", "spring", "burrel", "anna", "tres", "sweet", "uber",
    "homemade", "ann","best","j", "anite", "anitas", "anman", "angie", "angry", "simple", "difficult", "andy", "andrew", "ancient", "still", "another", "best", "go",
    "grant", "grandma", "amusement", "park", "instruction", "kitchen", "test", "ww", "almost", "empty", "dressing", "instant", "like", "le", "virtually",
    "home", "made", "guilt", "guilty", "delicious", "parfait", "forgotten", "forget", "forevermama", "diet", "can", "real", "former",
    "miss", "fabulous", "forever", "authentic", "fortnum", "mason", "kid", "foolproof", "football", "season", "diabetic",
    "two", "small", "one", "three", "four", "five", "thanksgiving", "dream", "foothill", "paula", "deen", "food", "processor", "safari", "processor",
    "traditional", "forbidden", "flavorful", "grandmag", "grandmama", "grandmaman", "grandma", "grandmom", "lena", "alicia", "alisa", "alice", "ali", "bit", "different",
    "eat", "family", "global", "gourmet", "yam", "yam", "emotional", "balance", "tonight", "feel", "cooking", "got", "birthday", "air", "way", "mr", "never", "weep", "half",
    "anything", "pour", "put", "fork", "say", "stove", "top", "thought", "prize", "winning", "add", "ad", "good", "better", "da", "style", "even", "bran", "fake", "fire", "beautiful"
    "l", "game", "day", "hate", "world", "minute", "type", "starbucks", "biggest", "dressed", "summertime", "elmer", "johnny", "depp", "c", "p", "h", "clove", "er", "star", "week",
    "affair", "elegant", "student", "z", "whole", "lotta", "w", "z", "b", "aaron", "craze", "a", "abc", "absolute", "absolut", "absolutely", "perfection", "delightful", "lazy", "morning",
    "abuelo", "abuelito", "abuelita", "abuela", "acadia", "accidental", "adam", "little", "interest", "addicting", "addie", "adele", "adelaide", "adi", "adie", "adriana",
    "adult", "affordable", "alison", "holst", "purpose", "allegheny", "allegedly", "original", "allergic", "ex", "allergy", "allergen", "allen", "poorman", "backyard",
    "alton", "brown", "whatever", "anthony", "anytime", "april", "fool", "ya", "fooled", "sandra", "lee", "edna", "emma", "emy", "evy", "eva", 'evelyn', "fannie", "fanny", "flo", "gladys", "helen", "grace", "ira", "irma",
    "isse", "jean", "janet", "jenny", "juju", "judy", "kathy", "kathi", "kellie", "kelly", "laura", "lee", "kay", "kathleen", "laura", "lee", "lesley", "lil", "linda", "liz", "lois", "louisse",
    "mag", 'martguerite', "margie", "marge", "maggie", "martha", "marylin", "marion", "mary", "marthy", "melody", "michel", "meda", "millie", "muriel", "myrna", "nelda", "nancy", "paulie", "phillis", "rae", "rebecca",
    "rose", "sadie", "sarah", "sara", "sue", "susan", "teresa", "theresa", "auntie", "em", "barbara", "barb", "irene", "lolo", "lori", "lu", "maebelle",
    "aunty", "aussie", "aurora", "austin", "l", "q"
    
    }
stop_words.update(additional) 

# Function to clean recipe names
def clean_recipe_names(recipes):
    cleaned_recipes = []
    
    for recipe in recipes:

        recipe = recipe.lower()
        recipe = re.sub(r'[^a-z\s]', '', recipe)
        
        recipe_words = recipe.split()
        
        # Lemmatize first
        recipe_words = [lemmatizer.lemmatize(word) for word in recipe_words]
        
        # Then remove stopwords
        recipe_words = [word for word in recipe_words if word not in stop_words]
        
        cleaned_recipe = " ".join(recipe_words)
        cleaned_recipes.append(cleaned_recipe)
    
    return cleaned_recipes


data['name'] = clean_recipe_names(data['name'])

data.dropna(subset=['name', 'description'], inplace=True)
data.reset_index(inplace=True)
data.head()

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/maxboc/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/maxboc/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,id,name,minutes,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,137739,arriba baked squash mexican,55,"[60-minutes-or-less, time-to-make, course, mai...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"[make a choice and proceed with recipe, depend...",autumn is my favorite time of year to cook! th...,"[winter squash, mexican seasoning, mixed spice...",7
1,31490,breakfast pizza,30,"[30-minutes-or-less, time-to-make, course, mai...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"[preheat oven to 425 degrees f, press dough in...",this recipe calls for the crust to be prebaked...,"[prepared pizza crust, sausage patty, eggs, mi...",6
2,112140,chili,130,"[time-to-make, course, preparation, main-dish,...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"[brown ground beef in large pot, add chopped o...",this modified version of 'mom's' chili was a h...,"[ground beef, yellow onions, diced tomatoes, t...",13
3,59389,alouette potato,45,"[60-minutes-or-less, time-to-make, course, mai...","[368.1, 17.0, 10.0, 2.0, 14.0, 8.0, 20.0]",11,[place potatoes in a large pot of lightly salt...,"this is a super easy, great tasting, make ahea...","[spreadable cheese with garlic and herbs, new ...",11
4,44061,amish tomato ketchup canning,190,"[weeknight, time-to-make, course, main-ingredi...","[352.9, 1.0, 337.0, 23.0, 3.0, 0.0, 28.0]",5,"[mix all ingredients& boil for 2 1 / 2 hours ,...",my dh's amish mother raised him on this recipe...,"[tomato juice, apple cider vinegar, sugar, sal...",8


Normalizing columns with string or list of strings

In [11]:
data["description"] = data["description"].apply(lambda x: x.lower())

columns = ["tags", "steps", "ingredients"]

for c in columns:
    data[c] = data[c].apply(lambda x : [s.lower() for s in x])

data.head()


,id,name,minutes,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,137739,arriba baked squash mexican,55,"[60-minutes-or-less, time-to-make, course, mai...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"[make a choice and proceed with recipe, depend...",autumn is my favorite time of year to cook! th...,"[winter squash, mexican seasoning, mixed spice...",7
1,31490,breakfast pizza,30,"[30-minutes-or-less, time-to-make, course, mai...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"[preheat oven to 425 degrees f, press dough in...",this recipe calls for the crust to be prebaked...,"[prepared pizza crust, sausage patty, eggs, mi...",6
2,112140,chili,130,"[time-to-make, course, preparation, main-dish,...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"[brown ground beef in large pot, add chopped o...",this modified version of 'mom's' chili was a h...,"[ground beef, yellow onions, diced tomatoes, t...",13
3,59389,alouette potato,45,"[60-minutes-or-less, time-to-make, course, mai...","[368.1, 17.0, 10.0, 2.0, 14.0, 8.0, 20.0]",11,[place potatoes in a large pot of lightly salt...,"this is a super easy, great tasting, make ahea...","[spreadable cheese with garlic and herbs, new ...",11
4,44061,amish tomato ketchup canning,190,"[weeknight, time-to-make, course, main-ingredi...","[352.9, 1.0, 337.0, 23.0, 3.0, 0.0, 28.0]",5,"[mix all ingredients& boil for 2 1 / 2 hours ,...",my dh's amish mother raised him on this recipe...,"[tomato juice, apple cider vinegar, sugar, sal...",8


In [12]:
""" No row to drop was found
print(len(data))
data.dropna(subset=columns, inplace=True, how="any")
len(data)
"""

' No row to drop was found\nprint(len(data))\ndata.dropna(subset=columns, inplace=True, how="any")\nlen(data)\n'

In [15]:
# create strings from string lists
data["steps_strings"] = data["steps"].apply(lambda x : ' '.join(x))

### Unit Management and Stemming

What we manage so far :
- Some of the mispelled terms (e.g. "minuet" instead of "minute") 
- Standardization for units (mL to L, °F to °C ...)
- Some weird notations like "2-3 kg", "2 to 3 cups" "2 1/2 hours" 

In [78]:
import re
import pandas as pd
from nltk.stem import PorterStemmer
from rapidfuzz import process, fuzz

# Define the stemmer
ps = PorterStemmer()
STANDARD_VOLUME = "liter"
STANDARD_TIME = "minute"
STANDARD_WEIGHT = "gram"

# Unit conversion factors to standardized units
unit_conversions = {
    # Volume conversions to liter
    "cup": 0.24,         # 1 cup = 0.24 liters
    "quart": 0.95,       # 1 quart = 0.95 liters
    "mL": 0.001,         # 1 mL = 0.001 liters
    
    # Time conversions to minutes
    "second": 1/60,      # 1 second = 1/60 minutes
    "hours": 60,         # 1 hour = 60 minutes
    "week": 10080,       # 1 week = 10080 minutes
    
    # Weight conversions to grams
    "pound": 453.59,     # 1 pound = 453.59 grams
    "ounce": 28.35,      # 1 ounce = 28.35 grams
    "kg": 1000,          # 1 kg = 1000 grams
    "g": 1,              # 1 g = 1 gram

    # Inch to centimeter conversion
    "inch": 2.54,        # 1 inch = 2.54 cm
    "inches": 2.54,      # 1 inch = 2.54 cm
}

# Mapping of units to their standard category
unit_categories = {
    # Volume units
    "cup": STANDARD_VOLUME,
    "quart": STANDARD_VOLUME,
    "mL": STANDARD_VOLUME,
    "liter": STANDARD_VOLUME,
    
    # Time units
    "second": STANDARD_TIME,
    "minutes": STANDARD_TIME,
    "hours": STANDARD_TIME,
    "week": STANDARD_TIME,
    
    # Weight units
    "pound": STANDARD_WEIGHT,
    "ounce": STANDARD_WEIGHT,
    "kg": STANDARD_WEIGHT,
    "g": STANDARD_WEIGHT,
    "gram": STANDARD_WEIGHT,

    # Measurement units
    "inch": "cm",
    "inches": "cm",
}



common_units = list(unit_categories.keys())

typo_corrections = {
    "gram": "g",
    "gm": "g",
    "lb": "pound",
    "oz": "ounce",
    "kilogram": "kg",
    "centimetr": "cm",
    "centimet": "cm",
    "mm": "mL",
    "millimet": "mL",
    "millilit": "mL",
    "centigrad": "°C",
    "litr": "liter",
    "cupof": "cup of",
    "talbespoon": "tablespoon",
    "tablespon": "tablespoon",
    "tablesppoon": "tablespoon",
    "tblpss": "tablespoon",
    "tbso": "tablespoon",
    "tbspn": "tablespoon",
    "tbslp": "tablespoon",
    "tsbp": "tablespoon",
    "tlbsp": "tablespoon",
    "tablestoon": "tablespoon",
    "tablepoon": "tablespoon",
    "teasppon": "teaspoon",
    "teapsoon": "teaspoon",
    "teaspon": "teaspoon",
    "teaspoom": "teaspoon",
    "cu": "cup",

    # Times
    "min" : "minutes",
    "minuet": "minutes",
    "minutesthen": "minutes",
    "minutesor": "minutes",
    "minutesyour": "minutes",
    "minuet": "minutes",
    "miniut": "minutes",
    "mimut": "minutes",
    "mionut": "minutes",
    "mintur": "minutes",
    "mkinut": "minutes",
    "mminut": "minutes",
    "munut": "minutes",
    "minuest": "minutes",
    "minunet": "minutes",
    "mintes": "minutes",
    "mutes": "minutes",
    "mutesr": "minutes",
    "minutesr": "minutes",
    "minuteslong": "minutes long",
    "minutesbrush": "minutes brush",
    "minnut": "minutes",
    "minuteuntil": "minutes until",
    "minutesm": "minutes",
    "nminut": "minutes",
    "minit": "minutes",
    "minutu": "minutes",
    "mihnut": "minutes",
    "mintut": "minutes",
    "minutr": "minutes",
    "ninut": "minutes",
    "minutew": "minutes",
    "minutess": "minutes",
    "minutesssssssss": "minutes",
    "minuteswil": "minutes will",
    "seccond": "second",
    "secong": "second",
    "seceond": "second",
    "housr": "hours",
    "houir": "hours",
    "hoursin": "hours",
    "hoursovernight": "hours overnight",
    "secon": "second",
    "seccond": "second",
    "secong": "second",
    "seceond": "second",
    "wk": "week",
    "hr": "hours",
    "b": "lb",
    "z": "oz",
    "″": "inch"
}

def correct_term(word):
    """
    This function apply a fuzzy matching 
    """
    # If it is a number return the word
    if not any(c.isalpha() for c in word):
        return word

    # Check if in mapping
    if word in typo_corrections:
        return typo_corrections[word]
      
    # Fuzzy matching
    match, score, _ = process.extractOne(word, common_units, scorer=fuzz.ratio)
    if score > 80:
        return match
    return word

def fahrenheit_to_celsius(f):
    return round((f - 32) * 5.0 / 9.0, 2)

def parse_range(word, next_word=None):
    """
    Detects numeric ranges like "2-3" or "2 to 3" and returns their mean as a float.
    E.g. => 2-3 kgs becomes 2.5 kgs
    """
    if re.match(r"^\d+(\.\d+)?-\d+(\.\d+)?$", word):  # "2-3"
        start, end = map(float, word.split("-"))
        return (start + end) / 2
    if next_word and word.isdigit() and next_word == "to":
        return "to" 
    return None

def correct_term(word):
    """
    This function apply a fuzzy matching 
    """
    # If it is a number return the word
    if not any(c.isalpha() for c in word):
        return word

    # Check if in mapping
    if word in typo_corrections:
        return typo_corrections[word]
      
    # Fuzzy matching
    match, score, _ = process.extractOne(word, common_units, scorer=fuzz.ratio)
    if score > 80:
        return match
    return word

def fahrenheit_to_celsius(f):
    return round((f - 32) * 5.0 / 9.0, 2)

def parse_range(word, next_word=None):
    """
    Detects numeric ranges like "2-3" or "2 to 3" and returns their mean as a float.
    E.g. => 2-3 kgs becomes 2.5 kgs
    """
    if re.match(r"^\d+(\.\d+)?-\d+(\.\d+)?$", word):  # "2-3"
        start, end = map(float, word.split("-"))
        return (start + end) / 2
    if next_word and word.isdigit() and next_word == "to":
        return "to" 
    return None

def standardize_units(text):
    words = text.lower().split()
    result_words = []
    i = 0
    while i < len(words):
        word = words[i]
        next_word = words[i+1] if i + 1 < len(words) else ""
        next2_word = words[i+2] if i + 2 < len(words) else ""
        next3_word = words[i+3] if i + 3 < len(words) else ""

        # Handle fractions like "1 / 2 inch"
        if (
            i + 2 < len(words)
            and re.match(r"^\d+(\.\d+)?$", word)
            and words[i+1] == "/"
            and re.match(r"^\d+(\.\d+)?$", words[i+2])
        ):
            numerator = float(word)
            denominator = float(words[i+2])
            fraction_value = numerator / denominator
            
            # Check if there's a unit after the fraction
            if i + 3 < len(words):
                corrected_unit = correct_term(words[i+3])
                if corrected_unit in unit_categories:
                    category = unit_categories[corrected_unit]
                    converted = fraction_value * unit_conversions.get(corrected_unit, 1)
                    result_words.append(f"{converted} {category}")
                    i += 4  # Move past the fraction and the unit
                    continue
            
            # If no unit or unrecognized unit, just keep the fraction as a decimal
            result_words.append(str(fraction_value))
            i += 3
            continue

        # Handle Fahrenheit to Celsius conversion - direct notation like "400F"
        fahrenheit_pattern = re.match(r"^(\d+(?:\.\d+)?)([°]?f)$", word)
        if fahrenheit_pattern:
            f = float(fahrenheit_pattern.group(1))
            c = fahrenheit_to_celsius(f)
            result_words.append(f"{c} °c")
            i += 1
            continue
            
        # Handle Fahrenheit to Celsius conversion
        if re.match(r"^\d+(\.\d+)?$", word):
            if next_word in {"f", "°f", "fahrenheit", "degrees f", "degree f"} or (next_word in {"degree", "degrees"} and next2_word in {"f", "°f", "fahrenheit"}):
                f = float(word)
                c = fahrenheit_to_celsius(f)
                result_words.append(f"{c} °c")
                
                # Skip the appropriate number of tokens
                if next_word in {"degree", "degrees"} and next2_word in {"f", "°f", "fahrenheit"}:
                    i += 3
                elif next_word in {"degrees f", "degree f"}:
                    i += 2
                else:
                    i += 2
                continue

        # Handle "2-3 kg"
        value = parse_range(word, next_word)
        if isinstance(value, float) and next_word:
            corrected_unit = correct_term(next_word)
            if corrected_unit in unit_categories:
                category = unit_categories[corrected_unit]
                converted = value * unit_conversions.get(corrected_unit, 1)
                result_words.append(f"{converted} {category}")
                i += 2
                continue

        # Handle "2 to 3 kg"
        if value == "to" and next2_word.replace('.', '', 1).isdigit() and next3_word:
            average = (float(word) + float(next2_word)) / 2
            corrected_unit = correct_term(next3_word)
            if corrected_unit in unit_categories:
                category = unit_categories[corrected_unit]
                converted = average * unit_conversions.get(corrected_unit, 1)
                result_words.append(f"{converted} {category}")
                i += 4
                continue

        # Handle regular value + unit
        if re.match(r"^\d+(\.\d+)?$", word) and next_word:
            corrected_unit = correct_term(next_word)
            if corrected_unit in unit_categories:
                category = unit_categories[corrected_unit]
                converted = float(word) * unit_conversions.get(corrected_unit, 1)
                result_words.append(f"{converted} {category}")
                i += 2
                continue

        # Default: stem and append
        result_words.append(ps.stem(correct_term(word)))
        i += 1

    return " ".join(result_words)

data["steps_string_standardize"] = data["steps_strings"].apply(standardize_units)

The following function is only here to compare what we had before, and what we have after the conversions

In [79]:
units = ["liter", "minute", "gram", "hours", "cup", "quart", "mL", "second", "week", 
         "pound", "ounce", "kg", "oz", "g", "°c", "°f", "fahrenheit", "degree", "celsius"]


# Function to find differences only on number + unit
def number_unit_diff(row):
    pattern = r'(\d+(?:\.\d+)?)\s*(' + '|'.join(units) + r')'

    matches1 = set(re.findall(pattern, row['steps_string_standardize']))
    matches2 = set(re.findall(pattern, row['steps_strings']))

    diff1 = matches1 - matches2
    diff2 = matches2 - matches1
    
    return {
        'only_in_standardize': list(diff1),
        'only_in_not_standardize': list(diff2)
    }

# Apply the function row-wise
diffs = data.apply(number_unit_diff, axis=1, result_type='expand')

# Concatenate
result = pd.concat([data, diffs], axis=1)

In [80]:
result[["id", "only_in_standardize", "only_in_not_standardize"]]

,id,only_in_standardize,only_in_not_standardize
0,137739,"[(40.0, minute)]","[(40, minute), (350, degree)]"
1,31490,"[(5.0, minute), (17.5, minute), (218.33, °c)]","[(5, minute), (425, degree), (20, minute)]"
2,112140,"[(150.0, minute), (420.0, minute)]","[(8, hours), (3, hours)]"
3,59389,"[(15.0, minute), (120.0, minute)]","[(15, minute), (2, hours)]"
4,44061,"[(30.0, minute)]","[(2, hours)]"
...,...,...,...
226652,486161,"[(5.0, minute), (30.0, minute)]","[(5, minute), (30, minute)]"
226653,493372,[],[]
226654,308080,[],[]
226655,298512,"[(204.44, °c), (10.0, minute), (7.0, minute)]","[(10, minute), (8, minute)]"


Test case with 1/2 inch

In [ ]:
print(data[data["id"] == 298512]["steps_strings"].iloc[0])
print(data[data["id"] == 298512]["steps_string_standardize"].iloc[0])

place melted butter in a large mixing bowl and add each remaining ingredient as listed , beating well after each addition to achieve a smooth dough let dough stand uncovered 10 minutes make golf-ball-sized shapes of the dough and insert a flat , wooden popsicle stick into one side of each ball carefully flatten each ball of dough to a 1 / 2 inch thick patty place 3 inches apart on greased baking sheets bake at 400f for 6-8 minutes or til just browned around the edges , but still white on tops of cookies remove from the oven and smear the top of each cookie generously with heated , canned , ready-to-spread frosting frostings may be tinted with few drops food coloring once it has been heated let cookies cool on baking sheet 10 minutes before carefully removing to paper towels
place melt butter in a larg mix bowl and add each remain ingredi as list , beat well after each addit to achiev a smooth dough let dough stand uncov 10.0 minute make golf-ball-s shape of the dough and insert a flat 

Test case with a mention of "350 degrees" but no unit => not managed yet 

In [83]:
137739
print(data[data["id"] == 137739]["steps_strings"].iloc[0])
print(data[data["id"] == 137739]["steps_string_standardize"].iloc[0])

make a choice and proceed with recipe depending on size of squash , cut into half or fourths remove seeds for spicy squash , drizzle olive oil or melted butter over each cut squash piece season with mexican seasoning mix ii for sweet squash , drizzle melted honey , butter , grated piloncillo over each cut squash piece season with sweet mexican spice mix bake at 350 degrees , again depending on size , for 40 minutes up to an hour , until a fork can easily pierce the skin be careful not to burn the squash especially if you opt to use sugar or butter if you feel more comfortable , cover the squash with aluminum foil the first half hour , give or take , of baking if desired , season with salt
make a choic and proceed with recip depend on size of squash , cut into half or fourth remov seed for spici squash , drizzl oliv oil or melt butter over each cut squash piec season with mexican season mix ii for sweet squash , drizzl melt honey , butter , grate piloncillo over each cut squash piec sea

In [90]:
pattern = r'(degree)'

# Filter rows
filtered_df = data[data['steps_strings'].str.contains(pattern, case=False, na=False, regex=True)]

# Display filtered rows
filtered_df.head()

/var/folders/pg/1l4d75p90r565jztmcw4v8c00000gn/T/ipykernel_2057/2248238086.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  filtered_df = data[data['steps_strings'].str.contains(pattern, case=False, na=False, regex=True)]


,id,name,minutes,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients,steps_strings,steps_string_standardize
0,137739,arriba baked squash mexican,55,"[60-minutes-or-less, time-to-make, course, mai...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"[make a choice and proceed with recipe, depend...",autumn is my favorite time of year to cook! th...,"[winter squash, mexican seasoning, mixed spice...",7,make a choice and proceed with recipe dependin...,make a choic and proceed with recip depend on ...
1,31490,breakfast pizza,30,"[30-minutes-or-less, time-to-make, course, mai...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"[preheat oven to 425 degrees f, press dough in...",this recipe calls for the crust to be prebaked...,"[prepared pizza crust, sausage patty, eggs, mi...",6,preheat oven to 425 degrees f press dough into...,preheat oven to 218.33 °c press dough into the...
6,67888,barbecued rib,120,"[weeknight, time-to-make, course, main-ingredi...","[1109.5, 83.0, 378.0, 275.0, 96.0, 86.0, 36.0]",10,[in a medium saucepan combine all the ingredie...,this recipe is posted by request and was origi...,"[pork spareribs, soy sauce, fresh garlic, fres...",22,in a medium saucepan combine all the ingredien...,in a medium saucepan combin all the ingredi fo...
7,75452,beat banana bread,70,"[weeknight, time-to-make, course, main-ingredi...","[2669.3, 160.0, 976.0, 107.0, 62.0, 310.0, 138.0]",12,"[preheat oven to 350 degrees, butter two 9x5"" ...",from ann hodgman's,"[sugar, unsalted butter, bananas, eggs, fresh ...",9,"preheat oven to 350 degrees butter two 9x5"" lo...","preheat oven to 350 degre butter two 9x5"" loaf..."
10,67547,bush baked bean,2970,"[weeknight, time-to-make, course, main-ingredi...","[462.4, 28.0, 214.0, 69.0, 14.0, 29.0, 23.0]",9,[in a very large sauce pan cover the beans and...,i'd have to say that this is a labor of love d...,"[great northern bean, chicken bouillon cubes, ...",13,in a very large sauce pan cover the beans and ...,in a veri larg sauc pan cover the bean and bou...


### Tokenization

In [76]:
data.columns

Index(['id', 'name', 'minutes', 'tags', 'nutrition', 'n_steps', 'steps',
       'description', 'ingredients', 'n_ingredients', 'steps_strings',
       'steps_string_standardize'],
      dtype='object')

In [34]:
import spacy
df = data[:100]
nlp = spacy.load("en_core_web_sm")
texts = df["steps_string_standardize"].astype(str).tolist()
docs = list(nlp.pipe(texts))
df["steps_tokens"] = [[token.text for token in doc] for doc in docs]

/var/folders/pg/1l4d75p90r565jztmcw4v8c00000gn/T/ipykernel_2057/2358607447.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["steps_tokens"] = [[token.text for token in doc] for doc in docs]


In [35]:
df.head()

,id,name,minutes,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients,steps_strings,steps_string_standardize,steps_tokens
0,137739,arriba baked squash mexican,55,"[60-minutes-or-less, time-to-make, course, mai...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"[make a choice and proceed with recipe, depend...",autumn is my favorite time of year to cook! th...,"[winter squash, mexican seasoning, mixed spice...",7,make a choice and proceed with recipe dependin...,make a choic and proceed with recip depend on ...,"[make, a, choic, and, proceed, with, recip, de..."
1,31490,breakfast pizza,30,"[30-minutes-or-less, time-to-make, course, mai...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"[preheat oven to 425 degrees f, press dough in...",this recipe calls for the crust to be prebaked...,"[prepared pizza crust, sausage patty, eggs, mi...",6,preheat oven to 425 degrees f press dough into...,preheat oven to 218.33 °c press dough into the...,"[preheat, oven, to, 218.33, °, c, press, dough..."
2,112140,chili,130,"[time-to-make, course, preparation, main-dish,...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"[brown ground beef in large pot, add chopped o...",this modified version of 'mom's' chili was a h...,"[ground beef, yellow onions, diced tomatoes, t...",13,brown ground beef in large pot add chopped oni...,brown ground beef in larg pot add chop onion t...,"[brown, ground, beef, in, larg, pot, add, chop..."
3,59389,alouette potato,45,"[60-minutes-or-less, time-to-make, course, mai...","[368.1, 17.0, 10.0, 2.0, 14.0, 8.0, 20.0]",11,[place potatoes in a large pot of lightly salt...,"this is a super easy, great tasting, make ahea...","[spreadable cheese with garlic and herbs, new ...",11,place potatoes in a large pot of lightly salte...,place potato in a larg pot of lightli salt wat...,"[place, potato, in, a, larg, pot, of, lightli,..."
4,44061,amish tomato ketchup canning,190,"[weeknight, time-to-make, course, main-ingredi...","[352.9, 1.0, 337.0, 23.0, 3.0, 0.0, 28.0]",5,"[mix all ingredients& boil for 2 1 / 2 hours ,...",my dh's amish mother raised him on this recipe...,"[tomato juice, apple cider vinegar, sugar, sal...",8,"mix all ingredients& boil for 2 1 / 2 hours , ...",mix all ingredients& boil for 2 1 / 120.0 minu...,"[mix, all, ingredients, &, boil, for, 2, 1, /,..."
